### Ce notebook permet de faire une vérification de la qualité des données avant toute consolidation, visualisation ou modélisation.

In [1]:
import pandas as pd
from pathlib import Path

###  Chargement de données

In [2]:
# Chemin d'accès aux données brutes

RAW_DATA_DIR = Path("../..") / "data" / "raw"

# Charger les datasets

files = {
    "idmc": "data_idmc_depuis_2000.csv",
    "solutions": "data_solutions_depuis_2000.csv",
    "decisions": "decisions_asile_depuis_2000.csv",
    "demandes": "demandes_asile_depuis_2000.csv",
    "demographie": "demographie_depuis_2000.csv",
    "unrwa": "unrwa_depuis_2000.csv",
    "pays": "countries.csv"
}


dfs = {
    name: pd.read_csv(RAW_DATA_DIR / filename)
    for name, filename in files.items()
}

for name, df in dfs.items():
    print(f"{name:15} : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

idmc            : 934 lignes × 10 colonnes
solutions       : 21,258 lignes × 13 colonnes
decisions       : 113,929 lignes × 17 colonnes
demandes        : 120,597 lignes × 14 colonnes
demographie     : 116,781 lignes × 24 colonnes
unrwa           : 296 lignes × 10 colonnes
pays            : 232 lignes × 16 colonnes


### Unicité : recherche des doublons stricts

In [6]:
for name, df in dfs.items():

    n_dup = df.duplicated().sum()
    pct_dup = 100 * n_dup / len(df) if len(df) else 0

    print(
        f"{name:15} "
        f"{n_dup:,} doublons stricts "
        f"({pct_dup:.2f} %)"
    )

idmc            0 doublons stricts (0.00 %)
solutions       0 doublons stricts (0.00 %)
decisions       0 doublons stricts (0.00 %)
demandes        0 doublons stricts (0.00 %)
demographie     0 doublons stricts (0.00 %)
unrwa           0 doublons stricts (0.00 %)
pays            0 doublons stricts (0.00 %)


In [7]:
doublons = dfs["idmc"][
    dfs["idmc"].duplicated(keep=False)
]

display(doublons)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,total


In [8]:
doublons = dfs["solutions"][
    dfs["solutions"].duplicated(keep=False)
]

display(doublons)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,returned_refugees,resettlement,naturalisation,returned_idps


In [9]:
doublons = dfs["decisions"][
    dfs["decisions"].duplicated(keep=False)
]

display(doublons)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,procedure_type,dec_level,dec_pc,dec_recognized,dec_other,dec_rejected,dec_closed,dec_total


In [10]:
doublons = dfs["demandes"][
    dfs["demandes"].duplicated(keep=False)
]

display(doublons)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,procedure_type,app_type,dec_level,app_pc,applied


In [11]:
doublons = dfs["demographie"][
    dfs["demographie"].duplicated(keep=False)
]

display(doublons)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,f_0_4,...,f_other,f_total,m_0_4,m_5_11,m_12_17,m_18_59,m_60,m_other,m_total,total


In [12]:
doublons = dfs["unrwa"][
    dfs["unrwa"].duplicated(keep=False)
]
display(doublons)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,total


In [13]:
doublons = dfs["pays"][
    dfs["pays"].duplicated(keep=False)
]
display(doublons)

,0,id,code,iso,iso2,name,nameOrigin,nameLong,nameShort,nameFormal,nationality,majorArea,region,nameFr,majorAreaFr,regionFr


### Recherche des doublons métier
*Deux lignes peuvent être différentes techniquement mais représenter potentiellement le même objet métier.*

In [32]:
# Demandes d'asile

key_demandes = [
    "year",
    "coo_id",
    "coa_id",
    "procedure_type",
    "app_type",
    "dec_level"
]

key = [c for c in key_demandes if c in dfs["demandes"].columns]

duplicates_business = (
    dfs["demandes"]
    .loc[lambda x: x.duplicated(key, keep=False)]
    .sort_values(key)
)

display(duplicates_business)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,procedure_type,app_type,dec_level,app_pc,applied
25027,2025,262,Unknown,UKN,UNK,44,Colombia,COL,COL,G,N,FI,C,89
69706,2025,262,Unknown,UKN,UNK,44,Colombia,COL,COL,G,N,FI,P,70


In [31]:
# Décision d'asile
key_decisions = [
    "year",
    "coo_id",
    "coa_id",
    "procedure_type",
    "dec_level"
]

key = [c for c in key_decisions if c in dfs["decisions"].columns]

duplicates_business = (
    dfs["decisions"]
    .loc[lambda x: x.duplicated(key, keep=False)]
    .sort_values(key)
)

display(duplicates_business)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,procedure_type,dec_level,dec_pc,dec_recognized,dec_other,dec_rejected,dec_closed,dec_total
93620,2014,3,Albania,ALB,ALB,114,Luxembourg,LUX,LUX,G,RA,C,0,0,0,17,17
95417,2014,3,Albania,ALB,ALB,114,Luxembourg,LUX,LUX,G,RA,P,0,0,0,26,26
93621,2014,164,Serbia and Kosovo: S/RES/1244 (1999),SRB,SRB,114,Luxembourg,LUX,LUX,G,RA,C,0,0,0,17,17
95420,2014,164,Serbia and Kosovo: S/RES/1244 (1999),SRB,SRB,114,Luxembourg,LUX,LUX,G,RA,P,0,0,0,27,27
93622,2015,3,Albania,ALB,ALB,114,Luxembourg,LUX,LUX,G,RA,C,0,0,0,16,16
95581,2015,3,Albania,ALB,ALB,114,Luxembourg,LUX,LUX,G,RA,P,0,0,0,19,19
93623,2015,133,Montenegro,MNE,MNE,114,Luxembourg,LUX,LUX,G,RA,C,0,0,0,5,5
95583,2015,133,Montenegro,MNE,MNE,114,Luxembourg,LUX,LUX,G,RA,P,0,0,0,5,5
93624,2015,164,Serbia and Kosovo: S/RES/1244 (1999),SRB,SRB,114,Luxembourg,LUX,LUX,G,RA,C,0,0,0,30,30
95584,2015,164,Serbia and Kosovo: S/RES/1244 (1999),SRB,SRB,114,Luxembourg,LUX,LUX,G,RA,P,0,0,0,45,45


In [33]:
# IDMC
key_idmc = [
    "year",
    "coo_id",
    "coa_id"
]

key = [c for c in key_idmc if c in dfs["idmc"].columns]

duplicates_business = (
    dfs["idmc"]
    .loc[lambda x: x.duplicated(key, keep=False)]
    .sort_values(key)
)

display(duplicates_business)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,total


In [34]:
# Solutions
key_solutions = [
    "year",
    "coo_id",
    "coa_id"
]

key = [c for c in key_solutions if c in dfs["solutions"].columns]

duplicates_business = (
    dfs["solutions"]
    .loc[lambda x: x.duplicated(key, keep=False)]
    .sort_values(key)
)

display(duplicates_business)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,returned_refugees,resettlement,naturalisation,returned_idps


In [36]:
# Demographie
key_demographie = [
    "year",
    "coo_id",
    "coa_id"
]
key = [c for c in key_demographie if c in dfs["demographie"].columns]

duplicates_business = (
    dfs["demographie"]
    .loc[lambda x: x.duplicated(key, keep=False)]
    .sort_values(key)
)

display(duplicates_business)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,f_0_4,...,f_other,f_total,m_0_4,m_5_11,m_12_17,m_18_59,m_60,m_other,m_total,total


In [37]:
# UNRAWA
key_unrwa = [
    "year",
    "coo_id",
    "coa_id"
]
key = [c for c in key_demographie if c in dfs["unrwa"].columns]

duplicates_business = (
    dfs["unrwa"]
    .loc[lambda x: x.duplicated(key, keep=False)]
    .sort_values(key)
)

display(duplicates_business)

,year,coo_id,coo_name,coo,coo_iso,coa_id,coa_name,coa,coa_iso,total


In [39]:
# Referentiel Pays
key = "id"

duplicates_business = (
    dfs["pays"]
    .loc[lambda x: x.duplicated(key, keep=False)]
    .sort_values(key)
)

display(duplicates_business)

,0,id,code,iso,iso2,name,nameOrigin,nameLong,nameShort,nameFormal,nationality,majorArea,region,nameFr,majorAreaFr,regionFr
